In [1]:
# 1. Загрузка данных, метрики, роли, кластеры и приоритеты
from pathlib import Path
import pandas as pd
import networkx as nx
from IPython.display import display
import starter
from starter import (
    load, build_graph, basic_features, assign_roles,
    assign_clusters_and_priority, write_outputs,
)

PROJECT_ROOT = Path(starter.__file__).resolve().parent
OUT_DIR = PROJECT_ROOT / "out"
edges, nodes, tx = load(PROJECT_ROOT / "data")
G = build_graph(edges)
df = assign_roles(basic_features(G, nodes), G)
df, clusters = assign_clusters_and_priority(df, G)
write_outputs(df, OUT_DIR, clusters)
print(f"Узлов: {len(df)}, рёбер: {G.number_of_edges()}, кластеров: {len(clusters)}")


Выгрузки записаны в /Users/aazanondakanova/Documents/hack-ee20e5a9-specialz/out/
Узлов: 2248, рёбер: 3119, кластеров: 87


In [2]:
# 2. Топ-25 узлов; gid читаем строкой для сохранения всех цифр
top_nodes = pd.read_csv(OUT_DIR / "top_nodes.csv", dtype={"gid": str})
display(top_nodes.head(25))


,rank,gid,role,priority_score,why
0,1,100000008346837100,coordinator,0.539263,PR_norm=0.0903 × 0.4; BC_norm=1.0000 × 0.3; ро...
1,2,100000003016635100,coordinator,0.519585,PR_norm=0.1964 × 0.4; BC_norm=0.8034 × 0.3; ро...
2,3,100000008603629100,coordinator,0.471094,PR_norm=0.1967 × 0.4; BC_norm=0.6414 × 0.3; ро...
3,4,100000006866783100,coordinator,0.459386,PR_norm=0.1436 × 0.4; BC_norm=0.6731 × 0.3; ро...
4,5,100000000706545100,consolidator,0.449424,PR_norm=0.5386 × 0.4; BC_norm=0.0022 × 0.3; ро...
5,6,100000005910114100,coordinator,0.428115,PR_norm=0.0597 × 0.4; BC_norm=0.6505 × 0.3; ро...
6,7,100000004603109100,consolidator,0.412158,PR_norm=0.5303 × 0.4; BC_norm=0.0001 × 0.3; ро...
7,8,100000003684369100,coordinator,0.411465,PR_norm=0.3334 × 0.4; BC_norm=0.2564 × 0.3; ро...
8,9,100000000331309100,coordinator,0.410029,PR_norm=0.1445 × 0.4; BC_norm=0.5074 × 0.3; ро...
9,10,100000002224132100,peripheral,0.400208,PR_norm=1.0000 × 0.4; BC_norm=0.0007 × 0.3; ро...


In [3]:
def inspect_flows(gid):
    key = str(gid).strip()
    matches = df.loc[df["gid"].astype(str).eq(key)]

    if matches.empty:
        print(f"gid {key} не найден")
        return

    row = matches.iloc[0]
    node = row["gid"]
    print(
        f"gid={key} | role={row['role']} | "
        f"priority={row['priority_score']:.4f}"
    )
    print(row["evidence"])

    incoming = pd.DataFrame([
        {"плательщик": str(src), "получатель": key,
         "сумма_KZT": attrs["sum_kzt"], "число_переводов": attrs["n_tx"]}
        for src, _, attrs in G.in_edges(node, data=True)
    ])
    outgoing = pd.DataFrame([
        {"плательщик": key, "получатель": str(dst),
         "сумма_KZT": attrs["sum_kzt"], "число_переводов": attrs["n_tx"]}
        for _, dst, attrs in G.out_edges(node, data=True)
    ])

    print(f"\nВходящие связи: {len(incoming)}")
    display(incoming.sort_values("сумма_KZT", ascending=False)
            if not incoming.empty else incoming)
    print(f"Исходящие связи: {len(outgoing)}")
    display(outgoing.sort_values("сумма_KZT", ascending=False)
            if not outgoing.empty else outgoing)

inspect_flows("100000002224132100")

gid=100000002224132100 | role=peripheral | priority=0.4002
in_deg=5, out_deg=4, in_kzt=3.95102e+06, pass_through=3.066; Нет совпадений

Входящие связи: 5


,плательщик,получатель,сумма_KZT,число_переводов
0,100000003835149100,100000002224132100,2900720.0,6
1,100000005664922100,100000002224132100,510000.0,1
4,100000004603109100,100000002224132100,300000.0,1
2,100000004892144100,100000002224132100,180300.0,1
3,100000000594783100,100000002224132100,60000.0,1


Исходящие связи: 4


,плательщик,получатель,сумма_KZT,число_переводов
2,100000002224132100,100000004892144100,4400000.0,6
3,100000002224132100,100000000594783100,2635000.0,10
0,100000002224132100,100000004603109100,2580000.0,6
1,100000002224132100,100000000552584100,2500000.0,3


In [4]:
cut = df[df["truncated_by_depth"]]
assert len(cut) == 444
assert cut["evidence"].str.contains("артефакт глубины", regex=False).all()
assert cut["role"].eq("terminal").all()
assert cut["role_score"].eq(0.5).all()
seeds = df[df["is_seed"]]
assert not seeds["role"].isin(["consolidator", "transit"]).any()
assert seeds["evidence"].str.contains("in_kzt занижен", regex=False).all()
assert df.loc[df["in_kzt"].eq(0), "pass_through"].isna().all()

In [5]:
inspect_flows("100000008346837100")  # координатор, № 1
inspect_flows("100000000706545100")  # консолидатор, № 5
inspect_flows("100000002224132100")  # высокий приоритет, роль peripheral

gid=100000008346837100 | role=coordinator | priority=0.5393
in_deg=9, out_deg=25, in_kzt=254359, pass_through=4.948; BC=0.00914 >= p95=0.000158

Входящие связи: 9


,плательщик,получатель,сумма_KZT,число_переводов
7,100000003796063100,100000008346837100,112000.0,1
0,100000004070318100,100000008346837100,60059.0,4
1,100000008003034100,100000008346837100,28500.0,1
8,100000004569484100,100000008346837100,25000.0,1
6,100000005048668100,100000008346837100,7000.0,1
2,100000004188160100,100000008346837100,6000.0,1
5,100000007593823100,100000008346837100,5800.0,1
3,100000008121806100,100000008346837100,5000.0,1
4,100000008379273100,100000008346837100,5000.0,1


Исходящие связи: 25


,плательщик,получатель,сумма_KZT,число_переводов
2,100000008346837100,100000005074393100,329133.0,2
19,100000008346837100,100000003050771100,229000.0,1
24,100000008346837100,100000008733194100,121344.0,1
1,100000008346837100,100000005910114100,92009.0,1
13,100000008346837100,100000008620570100,65000.0,1
15,100000008346837100,100000008571443100,40000.0,1
22,100000008346837100,100000008088082100,39138.0,1
11,100000008346837100,100000005053653100,33873.0,1
4,100000008346837100,100000006020215100,30000.0,1
5,100000008346837100,100000004351795100,29511.0,1


gid=100000000706545100 | role=consolidator | priority=0.4494
in_deg=3, out_deg=1, in_kzt=1.0703e+06, pass_through=0.07082; Средства накапливаются

Входящие связи: 3


...

,плательщик,получатель,сумма_KZT,число_переводов
0,100000000706545100,100000006901046100,75800.0,3


gid=100000002224132100 | role=peripheral | priority=0.4002
in_deg=5, out_deg=4, in_kzt=3.95102e+06, pass_through=3.066; Нет совпадений

Входящие связи: 5


,плательщик,получатель,сумма_KZT,число_переводов
0,100000003835149100,100000002224132100,2900720.0,6
1,100000005664922100,100000002224132100,510000.0,1
4,100000004603109100,100000002224132100,300000.0,1
2,100000004892144100,100000002224132100,180300.0,1
3,100000000594783100,100000002224132100,60000.0,1


Исходящие связи: 4


,плательщик,получатель,сумма_KZT,число_переводов
2,100000002224132100,100000004892144100,4400000.0,6
3,100000002224132100,100000000594783100,2635000.0,10
0,100000002224132100,100000004603109100,2580000.0,6
1,100000002224132100,100000000552584100,2500000.0,3


In [6]:
# Бонус: исходящие переводы через 1–2 календарных дня после входящих.
# Это временное соседство событий, а не установление происхождения денег.
# В данных есть даты без времени суток. Переводы в один день
# исключаем: их порядок установить нельзя.

events = tx.copy()
events["date"] = pd.to_datetime(events["date"]).dt.normalize()

incoming = events[["dst", "date", "sum_kzt"]].rename(
    columns={"dst": "gid", "date": "in_date", "sum_kzt": "in_amount"}
)
outgoing = events[["src", "date", "sum_kzt"]].rename(
    columns={"src": "gid", "date": "out_date", "sum_kzt": "out_amount"}
)

# Для каждого исходящего перевода ищем ближайшую предшествующую
# дату входящего перевода того же узла.
last_in_date = (
    incoming.groupby(["gid", "in_date"], as_index=False)
    .agg(n_in_on_date=("in_amount", "size"))
    .sort_values(["in_date", "gid"])
)

out_rows = (
    outgoing.reset_index(drop=True).rename_axis("out_event_id").reset_index()
    .sort_values(["out_date", "gid"])
)

# merge_asof требует глобальной сортировки по дате, а не сначала по gid.
assert out_rows["out_date"].is_monotonic_increasing
assert last_in_date["in_date"].is_monotonic_increasing

matched = pd.merge_asof(
    out_rows,
    last_in_date,
    left_on="out_date",
    right_on="in_date",
    by="gid",
    direction="backward",
    allow_exact_matches=False,
)

matched["days_after_in"] = (matched["out_date"] - matched["in_date"]).dt.days
assert len(matched) == len(out_rows)
assert matched["out_event_id"].is_unique
quick = matched[matched["days_after_in"].between(1, 2)].copy()
assert quick["out_event_id"].is_unique
assert (quick["in_date"] < quick["out_date"]).all()

temporal_summary = (
    quick.groupby("gid", as_index=False)
    .agg(
        n_out_1_2_days=("out_event_id", "size"),
        out_kzt_1_2_days=("out_amount", "sum"),
        example_in_date=("in_date", "min"),
    )
)

temporal_summary["gid"] = temporal_summary["gid"].astype(str)
temporal_summary = temporal_summary.merge(
    df[["gid", "role", "priority_score"]].assign(
        gid=lambda x: x["gid"].astype(str)
    ),
    on="gid",
    how="left",
)

temporal_summary = temporal_summary.sort_values(
    ["n_out_1_2_days", "out_kzt_1_2_days", "gid"],
    ascending=[False, False, True],
)

display(temporal_summary.head(20))
print("Узлов с исходящими через 1–2 дня после входящих:", len(temporal_summary))

,gid,n_out_1_2_days,out_kzt_1_2_days,example_in_date,role,priority_score
79,100000003016635100,124,7774721.0,2026-07-04,coordinator,0.519585
210,100000006866783100,84,3589613.0,2026-07-01,coordinator,0.459386
270,100000008603629100,68,4667113.0,2026-07-03,coordinator,0.471094
98,100000003684369100,63,8396847.0,2026-07-13,coordinator,0.411465
60,100000002578405100,53,1421355.0,2026-07-08,coordinator,0.243342
142,100000004400305100,51,2743335.0,2026-07-05,coordinator,0.312626
12,100000000437046100,43,5108524.0,2026-07-13,coordinator,0.325707
8,100000000331309100,37,6432803.0,2026-07-09,coordinator,0.410029
75,100000002957787100,29,796595.0,2026-07-01,coordinator,0.383377
10,100000000343175100,28,3386189.0,2026-07-01,coordinator,0.286239


Узлов с исходящими через 1–2 дня после входящих: 297


In [7]:
if not temporal_summary.empty:
    example_gid = temporal_summary.iloc[0]["gid"]
    display(
        quick[quick["gid"].astype(str).eq(example_gid)]
        [["gid", "in_date", "out_date", "days_after_in", "out_amount"]]
        .head(10)
    )

,gid,in_date,out_date,days_after_in,out_amount
574,100000003016635100,2026-07-04,2026-07-05,1.0,11075.0
575,100000003016635100,2026-07-04,2026-07-05,1.0,48150.0
576,100000003016635100,2026-07-04,2026-07-05,1.0,24075.0
577,100000003016635100,2026-07-04,2026-07-05,1.0,24075.0
578,100000003016635100,2026-07-04,2026-07-05,1.0,14666.0
579,100000003016635100,2026-07-04,2026-07-05,1.0,15000.0
580,100000003016635100,2026-07-04,2026-07-05,1.0,5297.0
694,100000003016635100,2026-07-05,2026-07-06,1.0,115200.0
695,100000003016635100,2026-07-05,2026-07-06,1.0,59000.0
696,100000003016635100,2026-07-05,2026-07-06,1.0,6014.0


In [8]:
# 3. Интерактивный граф; установка в окружение текущего ядра при необходимости
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyvis") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyvis"])

from pyvis.network import Network

ROLE_COLORS = {
    "consolidator": "red",
    "distributor": "orange",
    "transit": "yellow",
    "terminal": "blue",
    "coordinator": "purple",
    "peripheral": "gray",
}
node_rows = {str(row["gid"]): row for row in df.to_dict("records")}


def populate_network(net, graph, extra_nodes=()):
    # Строковые ID защищают 18-значные gid от округления JavaScript.
    for gid in dict.fromkeys([*graph.nodes, *extra_nodes]):
        row = node_rows[str(gid)]
        net.add_node(
            str(gid), label=str(gid), title=row["evidence"],
            color=ROLE_COLORS[row["role"]],
            size=18 if row["is_seed"] else 10,
        )
    for src, dst, attrs in graph.edges(data=True):
        net.add_edge(
            str(src), str(dst),
            title=f"sum_kzt={attrs['sum_kzt']:,.2f}; n_tx={attrs.get('n_tx', 0)}",
        )
    net.barnes_hut()
    net.set_options('{"physics": {"stabilization": {"iterations": 150}}}')
    return net


net = Network(directed=True, notebook=True, cdn_resources="in_line")
# Включаем также изоляты из df, которых нет в G.
populate_network(net, G, extra_nodes=df["gid"])
display(net.show("graph.html"))


graph.html

In [9]:
# 4. Поиск узла и направленный ego-граф радиуса 2
# nx.ego_graph по умолчанию следует исходящим рёбрам.
def search_node(gid):
    key = str(gid).strip()
    matches = df.loc[df["gid"].astype(str).eq(key)]
    if matches.empty:
        print(f"Узел {key} не найден. Передавайте gid строкой или целым числом.")
        return

    display(matches)
    graph_gid = matches["gid"].iloc[0]
    if graph_gid in G:
        ego = nx.ego_graph(G, graph_gid, radius=2)
    else:
        # Изолированные seed тоже доступны для поиска.
        ego = nx.DiGraph()
        ego.add_node(graph_gid)

    ego_net = Network(directed=True, notebook=True, cdn_resources="in_line")
    populate_network(ego_net, ego)
    ego_net.get_node(key)["size"] = 25
    display(ego_net.show(f"ego_{int(graph_gid)}.html"))


In [10]:
# 5. Пример: первый seed-узел
seed_gids = df.loc[df["is_seed"], "gid"]
if not seed_gids.empty:
    search_node(str(seed_gids.iloc[0]))
else:
    print("В данных нет seed-узлов.")


,gid,depth,is_seed,in_deg,out_deg,in_kzt,out_kzt,in_tx,out_tx,pagerank,...,role,role_score,evidence,cluster_id,seed_neighbor_fraction,pagerank_normalized,betweenness_normalized,role_priority,priority_score,why
0,100000000343175100,0,True,8,31,178712.0,4000039.0,14,33,0.000717,...,coordinator,0.8,"in_deg=8, out_deg=31, in_kzt=178712, pass_thro...",10,0.025641,0.117772,0.121888,1.0,0.286239,PR_norm=0.1178 × 0.4; BC_norm=0.1219 × 0.3; ро...


ego_100000000343175100.html
